# REINFORCE on CartPole-v1

Runnable implementation of the REINFORCE algorithm from the [policy-gradient lecture](/aiml-common/lectures/reinforcement-learning/policy-based-algorithms/reinforce). The agent learns a stochastic policy $\pi_\theta(a \mid s)$ over the two CartPole actions, using Monte Carlo returns $G_t$ as the unbiased estimator of $Q^{\pi_\theta}(s, a)$ in the policy gradient

$$
\nabla_\theta J(\pi_\theta) = \mathbb{E}_{\pi_\theta}\bigl[\nabla_\theta \log \pi_\theta(a \mid s)\, G_t\bigr].
$$

CartPole-v1 is considered solved at an average episode return of $475$ (max is $500$).

In [ ]:
%matplotlib inline

from pathlib import Path
import os

from torch.distributions import Categorical
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

gamma = 0.99
num_episodes = 300


_NB_NAME = "reinforce-cartpole.ipynb"


def _notebook_dir() -> Path:
    pm_input = os.environ.get("PM_INPUT_PATH")
    if pm_input and Path(pm_input).is_file():
        return Path(pm_input).resolve().parent
    for candidate in Path.cwd().rglob(_NB_NAME):
        return candidate.resolve().parent
    return Path.cwd()


IMAGES_DIR = _notebook_dir() / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

## Policy network and REINFORCE update

The policy is a small MLP with a single hidden layer of 64 units. `act` samples an action from the categorical distribution over the network's logits and stores `log π(a|s)` for the later gradient-ascent update. `train` computes the discounted return at each step (backward sweep for efficiency), forms the REINFORCE loss $-\sum_t \log \pi_\theta(a_t \mid s_t)\, G_t$, and performs one optimizer step per episode.

In [ ]:
class Pi(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.ReLU(),
            nn.Linear(64, out_dim),
        )
        self.onpolicy_reset()
        self.train()

    def onpolicy_reset(self):
        self.log_probs = []
        self.rewards = []

    def forward(self, x):
        return self.model(x)

    def act(self, state):
        x = torch.as_tensor(state, dtype=torch.float32)
        logits = self.forward(x)
        pd = Categorical(logits=logits)
        action = pd.sample()
        self.log_probs.append(pd.log_prob(action))
        return action.item()


def train_episode(pi, optimizer):
    T = len(pi.rewards)
    rets = np.empty(T, dtype=np.float32)
    future_ret = 0.0
    for t in reversed(range(T)):
        future_ret = pi.rewards[t] + gamma * future_ret
        rets[t] = future_ret
    rets = torch.as_tensor(rets, dtype=torch.float32)
    log_probs = torch.stack(pi.log_probs)
    loss = -(log_probs * rets).sum()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

## Training loop

CartPole is a very forgiving environment; a fresh, untuned REINFORCE agent typically reaches the 475-return solved threshold within a few hundred episodes. Rewards are stored per episode so we can plot the learning curve.

In [ ]:
env = gym.make("CartPole-v1")
in_dim = env.observation_space.shape[0]  # 4
out_dim = env.action_space.n              # 2
pi = Pi(in_dim, out_dim)
optimizer = optim.Adam(pi.parameters(), lr=0.01)

episode_returns = []
first_solved = None

for epi in range(num_episodes):
    state, _ = env.reset(seed=epi)
    for t in range(500):
        action = pi.act(state)
        state, reward, terminated, truncated, _ = env.step(action)
        pi.rewards.append(reward)
        if terminated or truncated:
            break
    loss = train_episode(pi, optimizer)
    total_reward = float(sum(pi.rewards))
    episode_returns.append(total_reward)
    if first_solved is None and total_reward > 475.0:
        first_solved = epi
    pi.onpolicy_reset()
    if epi % 25 == 0 or epi == num_episodes - 1:
        print(f"episode {epi:>3d}  loss={loss:>9.2f}  return={total_reward:>6.1f}")

env.close()
print(f"\nFirst episode with return > 475: {first_solved}")
print(f"Mean return over last 50 episodes: {np.mean(episode_returns[-50:]):.1f}")

## Learning curve

Per-episode returns (thin) and a 25-episode moving average (bold) show how quickly the policy climbs toward the 500 cap.

In [ ]:
returns = np.array(episode_returns, dtype=np.float32)
window = 25
if len(returns) >= window:
    smoothed = np.convolve(returns, np.ones(window) / window, mode="valid")
else:
    smoothed = returns

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(returns, alpha=0.35, color="#2563eb", label="episode return")
if len(smoothed) > 0:
    ax.plot(np.arange(len(smoothed)) + window - 1, smoothed, color="#0f172a", linewidth=2.0, label=f"{window}-episode moving avg")
ax.axhline(475, color="#059669", linestyle="--", linewidth=1, label="solved threshold (475)")
ax.set_xlabel("episode")
ax.set_ylabel("total return")
ax.set_title("REINFORCE on CartPole-v1 — learning curve")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMAGES_DIR / "learning-curve.svg", format="svg", bbox_inches="tight")
plt.close(fig)

![REINFORCE learning curve on CartPole-v1 — per-episode returns and a 25-episode moving average rising toward the 475 solved threshold](images/learning-curve.svg)

*Learning curve — per-episode return and moving average.*

## Where to go next

- **Reduce variance** by subtracting a state-dependent baseline $b(s)$ from the return. A common choice is $b(s) = v_\pi(s)$ estimated by a separate network; this produces the advantage $A(s, a) = G_t - v_\pi(s)$ and leads to actor-critic methods, PPO, and GRPO.
- **Try a harder environment** like LunarLander-v2; REINFORCE still works but convergence gets much noisier, motivating the baseline.
- Back to the lecture: [REINFORCE](/aiml-common/lectures/reinforcement-learning/policy-based-algorithms/reinforce).